In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import joblib

In [ ]:
data = {

    "job_id": [
        1, 2, 3, 4, 5
    ],

    "job_title": [
        "AI Engineer",
        "Frontend Developer",
        "Data Analyst",
        "Backend Developer",
        "Cyber Security Specialist"
    ],


    "required_skills": [

        "python machine learning deep learning tensorflow",

        "html css javascript react frontend",

        "python sql excel data analysis",

        "java python api database backend",

        "network security linux cybersecurity"
    ]

}


df = pd.DataFrame(data)
df.head()

In [ ]:
vectorizer = TfidfVectorizer()
skill_vectors = vectorizer.fit_transform(
    df["required_skills"]
)
print("Skill Gap AI Model trained successfully")

In [ ]:
def analyze_skill_gap(
    student_skills,
    job_index=0
):

    required = df.iloc[job_index]["required_skills"].split()

    current = [
        skill.lower()
        for skill in student_skills
    ]

    missing = list(
        set(required) - set(current)
    )

    student_text = " ".join(current)


    student_vector = vectorizer.transform(
        [student_text]
    )


    similarity = cosine_similarity(
        student_vector,
        skill_vectors[job_index]
    )[0][0]


    return {

        "job_title":
            df.iloc[job_index]["job_title"],

        "match_score":
            round(
                float(similarity),
                2
            ),

        "current_skills":
            current,

        "missing_skills":
            missing,

        "recommended_courses":
            [
                f"{skill} course"
                for skill in missing
            ]
    }


joblib.dump(
    vectorizer,
    "skillgap_vectorizer.pkl"
)


joblib.dump(
    skill_vectors,
    "skillgap_vectors.pkl"
)


joblib.dump(
    df,
    "skillgap_jobs.pkl"
)

print("Skill Gap AI Model saved successfully")

In [ ]:
student_skills = [
    "python",
    "sql",
    "machine learning"
]

result = analyze_skill_gap(
    student_skills,
    job_index=0
)

result

In [ ]:
from fastapi import FastAPI
import joblib
from sklearn.metrics.pairwise import cosine_similarity


app = FastAPI(
    title="Skill Gap Analysis AI API",
    description="Analyze missing skills using NLP",
    version="1.0"
)

vectorizer = joblib.load(
    "skillgap_vectorizer.pkl"
)


skill_vectors = joblib.load(
    "skillgap_vectors.pkl"
)

jobs = joblib.load(
    "skillgap_jobs.pkl"
)

print("Skill Gap AI Model loaded successfully")

In [ ]:
@app.get("/")
def home():

    return {
        "message": "Hello World"
    }


@app.post("/skill-gap")
def skill_gap(data: dict):

    student_id = data.get(
        "studentId"
    )
    student_skills = [
        skill.lower()
        for skill in data.get(
            "student_skills",
            []
        )
    ]

    job_index = data.get(
        "job_index",
        0
    )

    required_skills = jobs.iloc[
        job_index
    ]["required_skills"].split()

    student_text = " ".join(
        student_skills
    )


    student_vector = vectorizer.transform(
        [student_text]
    )

    similarity = cosine_similarity(
        student_vector,
        skill_vectors[job_index]
    )[0][0]



    missing_skills = list(
        set(required_skills)
        -
        set(student_skills)
    )


    return {

        "studentId":
            student_id,


        "job_title":
            jobs.iloc[job_index]["job_title"],


        "match_score":
            round(
                float(similarity),
                2
            ),


        "current_skills":
            student_skills,


        "missing_skills":
            missing_skills,


        "recommended_courses":
            [
                f"{skill} course"
                for skill in missing_skills
            ]
    }

In [ ]:
import nest_asyncio
import uvicorn


nest_asyncio.apply()


config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8001,
    log_level="info"
)


server = uvicorn.Server(config)


await server.serve()